# PKG geo metrics (F2): where customers are, how wide their payment footprint is, and where outside counterparties are likely to be

**PNC Treasury Management · DSI · Payment Knowledge Graph** · design: Claude Doc "PKG Geo Metrics (F2) — Design for
approval" (approved 2026-11-25), method from locatability v3 (`PKG_LOCATABILITY_MANIFEST_v3.md`) and Block F.

| table | one row per | holds |
|---|---|---|
| `pkg_cust_geo` | `mdm_id` (current state) | address and its quality grade, address move in the last 12 months, neighbour kernel, locating class, hub flag, an inferred location (separate columns) for customers without a usable address, cell |
| `pkg_cust_geo_monthly` | `mdm_id` × month (TM customers and organisations) | the 24 Block F columns, each twice: all counterparties and `_xh` without hubs |
| `pkg_cpty_geo` | outside counterparty entity (`ent_id`) | estimated point, cell (ZIP3, state), radius (r50, r90), probabilities within 50 / 250 km, evidence class |

QA files go to `OUT_DIR` (`radius_calibration.csv`, `gate_operating_points.csv`, `error_vs_k.csv`,
`truth_sensitivity.csv`, `footprint_bias.csv`, `hub_locating_registry.csv`, `zip3_centres.csv`, `run_log.csv`).

**Rules carried from August.** Coordinates are strings in MDM: screened before casting, never downcast. Ids stay
strings. A placeholder coordinate (`PLACEHOLDER_MIN`+ parties, or several ZIP5 on one point) is used nowhere. Observed and inferred
locations never merge. A location is never a bare point: every estimate carries a radius and a probability, and
`NO_EVIDENCE` stays empty rather than falling back to a footprint centre. Every accuracy figure is split by whether
the registered address is representative (`geo_registered_vs_flow_km` ≤ 25 km), and quoted with the in-core vs
out-of-core gap. The monthly block only locates counterparties that are PNC customers with a usable address
(observed); outside counterparties have no address.

**Run:** Run All. The monthly block is built month by month with markers (a rerun resumes); everything else is
rebuilt for the latest month on every run. First run: `MONTHS_BACK = 3` for review, then `None` for the full history.

## Settings

In [ ]:
# ---- Settings ------------------------------------------------------------------------------------
DB = "bdahd01p_dlcdi1_cdi_tm"
REF = "_v5"                                           # reference / metrics tables of the rebuild pass
T_ADDR   = "dsihd01p_dsi.neo4j_address"               # MDM address table, one row per party per daily snapshot (load_dt)
T_CUST   = f"{DB}.pkg_cust_dim{REF}"
T_EDGE_M = f"{DB}.pkg_edge_monthly"
T_MAP    = f"{DB}.pkg_cpty_entity_map{REF}"
T_ENT    = f"{DB}.pkg_cpty_entity_dim{REF}"
T_CPM    = f"{DB}.pkg_cpty_metrics_monthly{REF}"
OUT_TABLES = {"cust": f"{DB}.pkg_cust_geo", "monthly": f"{DB}.pkg_cust_geo_monthly", "cpty": f"{DB}.pkg_cpty_geo"}
OUT_DIR = "../metrics/geo"                            # QA files, next to the notebook

MONTHS_BACK   = 3              # monthly block: only the latest this many months (None = every published month)
LAST_MONTH    = None           # None = latest month published by pkg_edges_refresh
ADDR_SNAPSHOT = None           # None = latest load_dt of T_ADDR
MOVE_LOOKBACK_DAYS = 365       # moved_12m compares with the latest snapshot at least this many days older
MOVE_MIN_KM   = 5.0            # a move must be this far (the cleansing vendor re-geocodes in place by a few hundred metres)

# ---- address quality (D2)
COLOC_MIN       = 5            # parties on one coordinate (5 decimals, whole snapshot) → shared_building (a house or an
                               # office building: a real place, still usable); with several ZIP5 on it → placeholder
PLACEHOLDER_MIN = 100          # parties on one coordinate → placeholder (PNC towers, registered agents, lockboxes,
                               # address vendors). Exception: a PO box point with one ZIP5 (a post office) stays po_box
PO_REC_TYPES    = ["PostOfficeBox", "GeneralDelivery"]   # addr_loc_rec_type values of a post-office geocode
MILITARY_STATES = ["AE", "AP", "AA"]                     # APO / DPO / FPO: no usable location → foreign
LOW_PRECISION_DP = 2           # ≤ this many decimals ≈ 1 km: low precision (still usable)
GEO_OK_STATUSES = ["ok", "shared_building", "low_precision", "po_box"]   # usable as a location and in kernels
TRUTH_STATUSES  = ["ok", "shared_building", "low_precision"]             # self-scoring targets (a PO box point is the
                                                                         # post office, not the customer)
US_COUNTRIES    = ["US", "USA", "UNITED STATES", "UNITED STATES OF AMERICA", ""]
PO_BOX_RE   = r"(^| )(P ?O ?BOX|POST OFFICE BOX|PO BX|BOX \d)"
REG_AGENT_RE = r"(REGISTERED AGENT|CORPORATION SERVICE|CT CORPORATION|NATIONAL REGISTERED|INCORP SERVICES|LEGALZOOM|COGENCY|VCORP|NORTHWEST REGISTERED|CSC )"
CARE_OF_RE  = r"(^| )(C/O|C O |ATTN|PMB)"
PNC_RE      = r"(^| )PNC( |$)"

# ---- monthly block (D3, D4)
MONTHLY_TM_ORG_ONLY = True     # rows for TM customers and organisations (as the v5 pair table); every located customer
                               # still counts as a counterparty
HUB_MULT     = 500             # hub = counterparty whose degree (customers this month) ≥ HUB_MULT × the median degree
MIN_CP_SPREAD  = 2             # spread NULL below this many located counterparties
MIN_CP_ENTROPY = 5             # entropy NULL below this many
LOCALITY_KM  = [50, 250, 1000] # LOCAL < 50 · REGIONAL < 250 · MULTI_MARKET < 1000 · NATIONAL

# ---- kernels, self-scoring, locate() (D5, D6)
KERNEL_MONTHS = 12             # trailing window ending at the latest month
MIN_CP_BW     = 2              # kernel width NULL below this many located counterparties (after leave-one-out)
MIN_CP_BW_TRUST = 5            # below this the width exists but the neighbour is not used
COLOC_WIDTH_KM = 0.1           # a kernel narrower than this is a co-located cluster: not used
K_MAX        = 20              # tightest K_MAX usable neighbours are combined (inverse-variance)
CALIB_BINS   = 20              # predicted-width bins of the radius calibration (quantiles)
REP_GAP_KM   = 25              # pin_representative: registered point within this of the flow centre
CLASS_KM     = {"PIN": 25, "METRO": 100, "REGION": 400}   # evidence / locating class by median error (r50)
CLASS_MIN_PAIRS = 5            # a customer gets a locating class with at least this many scored pairs as a neighbour
GATE_PRECISION = 0.70          # gate: widest predicted width whose hit@50 precision stays ≥ this
CORE_SHARE   = 0.80            # core states = the fewest states holding this share of located customers
SCORE_PCT    = 100             # % of target customers (hash) self-scored (lower it if the step is too slow)
SEED = 17

# ---- v1.2: calibration of outside entities on outside entities (view-matched)
OUTSIDE_CALIBRATION = True     # pkg_cpty_geo radii / probabilities / class from outside entities that are the same
                               # business as a PNC organisation customer (validation 2026-11-26: the customer calibration
                               # was ~30 points too confident for outside entities)
MATCH_NAME_CLASSES = ["business"]
LEGAL_WORDS = ["INC", "INCORPORATED", "LLC", "L L C", "LC", "CORP", "CORPORATION", "CO", "COMPANY", "LTD", "LIMITED",
               "LP", "LLP", "PLLC", "PC", "PA", "THE", "NA", "N A", "DBA", "GROUP", "HOLDINGS"]
KEY_MIN_CHARS = 6
OUT_NB_BUCKETS = [1, 2, 5]     # PNC-neighbour buckets: 1 · 2–4 · 5+
OUT_BINS = 10                  # predicted-width bins per bucket (fewer when the bucket is small)
OUT_MIN_PER_BIN = 300          # calibration pairs per bin at least
OUT_MIN_TOTAL = 2000           # fewer calibration pairs than this: keep the customer calibration (with a warning)

# ---- run checks (D in §6)
JOIN_MIN     = 0.95            # edge customers found in the address snapshot
LEAK_FLAG_KM = 3               # representative median error below this → leave-one-out suspected broken (a target left in
                               # its neighbours' clouds pulls the error toward 0; a working one sits near the address scatter)
R90_COVER    = (0.80, 0.97)    # realised share inside r90 on the held-out half

# ---- cluster (no effect on the numbers)
SHUFFLE_PARTITIONS = 2000
RESOURCE_PROFILE = "acc"       # "acc" = inside the ACC limits; "standard" = 8g + 3g, 2 cores (over: nights / weekends only)
RESOURCES = {
    "acc":      {"spark.executor.memory": "4g", "spark.executor.memoryOverhead": "1g", "spark.executor.cores": "1",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "2g"},
    "standard": {"spark.executor.memory": "8g", "spark.executor.memoryOverhead": "3g", "spark.executor.cores": "2",
                 "spark.driver.memory": "4g", "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g",
                 "spark.dynamicAllocation.maxExecutors": "512", "spark.driver.maxResultSize": "4g"},
}
ACC_LIMITS = {"spark.executor.memory": "4g", "spark.executor.memoryOverhead": "1g", "spark.driver.memory": "4g",
              "spark.yarn.am.memory": "4g", "spark.yarn.am.memoryOverhead": "1g", "spark.dynamicAllocation.maxExecutors": "512"}
HDFS_REPLICATION = 2
NEW_SESSION_EACH_MONTH = True  # fresh Spark session (new YARN application) after every month of the monthly block
GEO_VERSION = "geo-v1.2"
MONTHLY_VERSION = "geo-v1.0.1"   # the monthly block's signature uses this: v1.2 changes nothing there (no rebuild)

## Setup

In [ ]:
# ---- Setup ---------------------------------------------------------------------------------------
import numpy as np
import pandas as pd
import warnings
for _alias, _target in {"object0": np.object_, "bool8": np.bool_, "int0": np.intp, "uint0": np.uintp}.items():
    if not hasattr(np, _alias):
        setattr(np, _alias, _target)
if "bool" not in np.__dict__:
    np.bool = np.bool_
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items
for _cat in (ResourceWarning, DeprecationWarning, FutureWarning):
    warnings.filterwarnings("ignore", category=_cat)
warnings.filterwarnings("ignore")

import os, math, time, json, hashlib, threading, datetime as dt, html as _html
from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F

_b = SparkSession.builder.appName("pkg_geo_metrics").config("spark.yarn.queue", "root.ua.CIB-AMG.spark")
for _k, _v in RESOURCES[RESOURCE_PROFILE].items():
    _b = _b.config(_k, _v)
_b = (_b.config("spark.hadoop.dfs.replication", str(HDFS_REPLICATION))
        .config("spark.dynamicAllocation.enabled", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("spark.sql.execution.arrow.pyspark.enabled", "true")
        .config("spark.sql.adaptive.enabled", "true").config("spark.sql.adaptive.skewJoin.enabled", "true")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.ui.retainedJobs", "100").config("spark.ui.retainedStages", "100")
        .config("spark.ui.retainedTasks", "10000").config("spark.sql.ui.retainedExecutions", "20")
        .enableHiveSupport())
_QUIET = ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage", "org.apache.kafka",
          "org.apache.spark.sql.catalyst.util.package", "org.apache.spark.network", "io.netty")

def start_session():
    s = _b.getOrCreate()
    s.conf.set("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
    s.conf.set("spark.sql.ansi.enabled", "false")
    s.sparkContext.setLogLevel("ERROR")
    try:
        _L = s._jvm.org.apache.logging.log4j
        for _n in _QUIET:
            _L.core.config.Configurator.setLevel(_n, _L.Level.OFF)
    except Exception:
        try:
            _L1 = s._jvm.org.apache.log4j
            for _n in _QUIET:
                _L1.LogManager.getLogger(_n).setLevel(_L1.Level.OFF)
        except Exception:
            pass
    s._jsc.hadoopConfiguration().set("dfs.replication", str(HDFS_REPLICATION))
    return s

spark = start_session()
HCONF = spark._jsc.hadoopConfiguration()

def new_session():
    """Stop the Spark session and start a fresh one (a new YARN application): nothing piles up in the driver."""
    global spark, HCONF
    try:
        spark.catalog.clearCache()
    except Exception:
        pass
    spark.stop()
    import gc; gc.collect()
    spark = start_session()
    HCONF = spark._jsc.hadoopConfiguration()

pd.set_option("display.max_rows", 300); pd.set_option("display.max_columns", 80)

# ---- one consolidated HTML output per cell
def tbl(pdf, pct=(), dec=()):
    pdf = pdf.copy()
    fmt = {}
    for c in pdf.columns:
        if c in pct:                                   fmt[c] = "{:.1%}"
        elif c in dec:                                 fmt[c] = "{:,.2f}"
        elif pd.api.types.is_bool_dtype(pdf[c]):       continue
        elif pd.api.types.is_integer_dtype(pdf[c]):    fmt[c] = "{:,}"
        elif pd.api.types.is_float_dtype(pdf[c]):      fmt[c] = "{:,.1f}"
    sty = pdf.style.format(fmt, na_rep="—")
    try:
        sty = sty.hide(axis="index")
    except AttributeError:
        sty = sty.hide_index()
    sty = sty.set_table_styles([{"selector": "th, td", "props": "padding: 2px 8px; font-size: 12px;"}])
    return sty.to_html()
def sec(title, body=""):
    return f"<h4 style='margin:14px 0 4px 0'>{_html.escape(title)}</h4>{body}"
def para(text, color=None):
    st = f"; color:{color}" if color else ""
    return f"<p style='margin:4px 0; font-size:13px{st}'>{_html.escape(text)}</p>"
def out(*parts):
    display(HTML("<div>" + "".join(p for p in parts if p) + "</div>"))

# ---- HDFS through the JVM (HCONF is looked up at call time, so a new session is picked up)
JPath = spark._jvm.org.apache.hadoop.fs.Path
def fs_for(p):
    return JPath(p).getFileSystem(HCONF)
def exists(p):
    return fs_for(p).exists(JPath(p))
def delete(p):
    return fs_for(p).delete(JPath(p), True) if exists(p) else False
def rename(a, b):
    return fs_for(a).rename(JPath(a), JPath(b))
def ls_dirs(p):
    if not exists(p):
        return []
    return [s.getPath().getName() for s in fs_for(p).listStatus(JPath(p)) if s.isDirectory()]
def size_gb(p):
    return fs_for(p).getContentSummary(JPath(p)).getLength() / 1e9 if exists(p) else 0.0
def write_json(p, obj):
    s = fs_for(p).create(JPath(p), True)
    s.write(bytearray(json.dumps(obj, default=str).encode())); s.close()
def read_json(p):
    if not exists(p):
        return None
    s = fs_for(p).open(JPath(p))
    try:
        return json.loads(spark._jvm.org.apache.commons.io.IOUtils.toString(s, "UTF-8"))
    finally:
        s.close()
def table_exists(t):
    db, name = t.split(".")
    return spark.sql(f"SHOW TABLES IN {db} LIKE '{name}'").count() > 0
def table_location(t):
    return [r[1] for r in spark.sql(f"DESCRIBE TABLE EXTENDED {t}").collect() if r[0] == "Location"][0].rstrip("/")
def md5(o):
    return hashlib.md5(json.dumps(o, sort_keys=True, default=str).encode()).hexdigest()[:12]
def month_add(m, k):
    t = int(m[:4]) * 12 + int(m[5:7]) - 1 + k
    return f"{t // 12:04d}-{t % 12 + 1:02d}"
def month_range(a, b):
    r, m = [], a
    while m <= b:
        r.append(m); m = month_add(m, 1)
    return r

# ---- geometry (unit vectors on the sphere; distances by chord, stable at short range)
R_EARTH_KM = 6371.0088
def unit(lat, lon):
    la, lo = F.radians(lat), F.radians(lon)
    return F.cos(la) * F.cos(lo), F.cos(la) * F.sin(lo), F.sin(la)
def chord_km(x1, y1, z1, x2, y2, z2):
    c = F.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2 + (z1 - z2) ** 2)
    return 2 * R_EARTH_KM * F.asin(F.least(F.lit(1.0), c / 2))
def latlon(x, y, z):
    return F.degrees(F.atan2(z, F.sqrt(x * x + y * y))), F.degrees(F.atan2(y, x))
def width_km(rbar):
    return R_EARTH_KM * F.sqrt(F.greatest(F.lit(0.0), 2 * (1 - rbar)))

DB_LOC = [r[1] for r in spark.sql(f"DESCRIBE DATABASE {DB}").collect() if r[0].lower() == "location"][0].rstrip("/")
LOC = {k: f"{DB_LOC}/{t.split('.')[1]}" for k, t in OUT_TABLES.items()}
WORK = f"{DB_LOC}/_tmp_pkg_geo"
os.makedirs(OUT_DIR, exist_ok=True)

# ---- what the session really got
def _gb(v):
    v = str(v).strip().lower()
    return float(v[:-1]) / (1024 if v.endswith("m") else 1) if v[-1:] in ("g", "m") else float(v)
_conf = dict(spark.sparkContext.getConf().getAll())
_rr = []
for _k in sorted(set(RESOURCES[RESOURCE_PROFILE]) | set(ACC_LIMITS)):
    _use = _conf.get(_k, "(default)")
    _lim = ACC_LIMITS.get(_k)
    _over = (_lim is not None and _use != "(default)"
             and (_gb(_use) > _gb(_lim) if _k != "spark.dynamicAllocation.maxExecutors" else int(_use) > int(_lim)))
    _rr.append({"setting": _k, "profile": RESOURCES[RESOURCE_PROFILE].get(_k, ""), "in use": _use,
                "ACC limit": _lim or "", "over ACC": "YES" if _over else ""})
_rr = pd.DataFrame(_rr)
_miss = _rr[(_rr.profile != "") & (_rr["in use"] != _rr.profile)]
out(sec(f"Spark {spark.version} · profile {RESOURCE_PROFILE} · pandas {pd.__version__} · numpy {np.__version__}"),
    para(f"The session was created before this cell: {len(_miss)} setting(s) differ from the profile. Restart the "
         "kernel and run this notebook from the top.", "#a00") if len(_miss) else "",
    tbl(_rr))

## §1 Inputs and plan

Column and id-type checks, the address snapshots, the published edge months and which months of the monthly block are
already built with the current settings (marker in the month folder).

In [ ]:
# §1  Inputs and plan
def need(table, cols):
    dt_ = dict(spark.table(table).dtypes)
    miss = [c for c in cols if c not in dt_]
    assert not miss, f"{table}: missing {miss}; has {sorted(dt_)}"
    return dt_

ADDR_COLS = ["mdm_id", "mdm_address_id", "latitude_degrees", "longitude_degrees", "zip_cd", "state_or_province", "city",
             "addr_country", "addr_loc_rec_type", "addr_line_1", "addr_line_2", "addr_cleansed_date", "load_dt"]
adt = need(T_ADDR, ADDR_COLS)
cdt = need(T_CUST, ["mdm_id", "party_type", "is_tm", "naics2", "name_group", "is_pnc_own"])
edt = need(T_EDGE_M, ["src_id", "src_kind", "dst_id", "dst_kind", "is_self", "amount", "month"])
mdt = need(T_MAP, ["cpty_id", "ent_l2"])
ndt = need(T_ENT, ["ent_id", "ent_name", "name_class"])
pdt = need(T_CPM, ["ent_id", "hub_tier", "month"])
bad = {f"{t}.{c}": d[c] for t, d, cols in [(T_ADDR, adt, ["mdm_id"]), (T_CUST, cdt, ["mdm_id"]),
                                           (T_EDGE_M, edt, ["src_id", "dst_id"]), (T_MAP, mdt, ["cpty_id", "ent_l2"])]
       for c in cols if d[c] != "string"}
assert not bad, f"id columns must be string (2026-07 incident): {bad}"

def partitions(table, key):
    vals = set()
    try:
        for r in spark.sql(f"SHOW PARTITIONS {table}").collect():
            for kv in r[0].split("/"):
                if kv.startswith(key + "="):
                    vals.add(kv.split("=", 1)[1])
    except Exception:
        vals = {str(r[0]) for r in spark.table(table).select(key).distinct().collect()}
    return sorted(v for v in vals if v and v != "None")

SNAPS = partitions(T_ADDR, "load_dt")
SNAP = ADDR_SNAPSHOT or SNAPS[-1]
assert SNAP in SNAPS, f"ADDR_SNAPSHOT {SNAP} not in {T_ADDR}"
def _pdate(s):
    d_ = "".join(ch for ch in str(s) if ch.isdigit())[:8]
    return dt.date(int(d_[:4]), int(d_[4:6]), int(d_[6:8]))
_snap_d = _pdate(SNAP)
_older = [s for s in SNAPS if _pdate(s) <= _snap_d - dt.timedelta(days=MOVE_LOOKBACK_DAYS)]
SNAP_PREV = _older[-1] if _older else None

E_LOC = table_location(T_EDGE_M)
EDGE_MONTHS = [m for m in partitions(T_EDGE_M, "month") if exists(f"{E_LOC}/month={m}/_PKG_BUILD")]
LAST = LAST_MONTH or EDGE_MONTHS[-1]
assert LAST in EDGE_MONTHS, f"LAST_MONTH {LAST} is not a published edge month"
ALL_MONTHS = [m for m in EDGE_MONTHS if m <= LAST]
RUN_MONTHS = ALL_MONTHS[-MONTHS_BACK:] if MONTHS_BACK else ALL_MONTHS
K_MONTHS = [m for m in ALL_MONTHS if m > month_add(LAST, -KERNEL_MONTHS)]
CPM_MONTHS = partitions(T_CPM, "month")
HUB_MONTH = max([m for m in CPM_MONTHS if m <= LAST], default=None)

# the monthly block's signature: settings that change its numbers (not the address snapshot: months keep the
# locations they were built with; a new snapshot is used by the months built after it)
SIG = md5({"v": MONTHLY_VERSION, "geo_ok": GEO_OK_STATUSES, "coloc": COLOC_MIN, "ph": PLACEHOLDER_MIN, "po": PO_REC_TYPES,
           "mil": MILITARY_STATES, "dp": LOW_PRECISION_DP, "hub": HUB_MULT,
           "sp": MIN_CP_SPREAD, "en": MIN_CP_ENTROPY, "loc": LOCALITY_KM, "tmorg": MONTHLY_TM_ORG_ONLY})
def marker(m):
    return read_json(f"{LOC['monthly']}/month={m}/_GEO_BUILD")
def done(m):
    mk = marker(m)
    return bool(mk) and mk.get("sig") == SIG and exists(f"{LOC['monthly']}/month={m}/_SUCCESS")
TODO = [m for m in RUN_MONTHS if not done(m)]

# edge customers of the latest month found in the address snapshot (the 2026-07 dtype incident produced plausible
# output with a near-zero join; this stops the run instead)
_ec = (spark.table(T_EDGE_M).where(F.col("month") == LAST)
         .select(F.explode(F.array(F.when(F.col("src_kind") == "CUST", F.col("src_id")),
                                   F.when(F.col("dst_kind") == "CUST", F.col("dst_id")))).alias("mdm_id"))
         .where(F.col("mdm_id").isNotNull()).select(F.trim("mdm_id").alias("mdm_id")).distinct())
_as = (spark.table(T_ADDR).where(F.col("load_dt").cast("string") == SNAP)
         .select(F.trim(F.col("mdm_id")).alias("mdm_id")).distinct())
JOIN_N = _ec.count()
JOIN_HIT = _ec.join(_as, "mdm_id", "left_semi").count()
JOIN_RATE = JOIN_HIT / max(JOIN_N, 1)

out(sec(f"§1 Inputs · address snapshot {SNAP} (previous for moves: {SNAP_PREV or 'none'}) · latest month {LAST}"),
    tbl(pd.DataFrame([
        {"item": "address snapshots", "value": f"{len(SNAPS)} ({SNAPS[0]} → {SNAPS[-1]})"},
        {"item": "published edge months", "value": f"{len(EDGE_MONTHS)} ({EDGE_MONTHS[0]} → {EDGE_MONTHS[-1]})"},
        {"item": "kernel window", "value": f"{K_MONTHS[0]} → {K_MONTHS[-1]} ({len(K_MONTHS)} months)"},
        {"item": "hub_tier month (outside entities)", "value": HUB_MONTH or "none"},
        {"item": "monthly block: months in scope", "value": f"{len(RUN_MONTHS)} ({RUN_MONTHS[0]} → {RUN_MONTHS[-1]})"},
        {"item": "monthly block: to build now", "value": f"{len(TODO)}" + (f" ({TODO[0]} → {TODO[-1]})" if TODO else "")},
        {"item": f"edge customers of {LAST} in the address snapshot", "value": f"{JOIN_HIT:,} of {JOIN_N:,} ({JOIN_RATE:.1%})"},
        {"item": "signature", "value": SIG}])))
assert JOIN_RATE >= JOIN_MIN, f"only {JOIN_RATE:.1%} of {LAST}'s edge customers are in address snapshot {SNAP} (min {JOIN_MIN:.0%}): check id types and the snapshot"

## §2 Customer locations (`pkg_cust_geo`, address part) and ZIP3 centres

One row per customer of `pkg_cust_dim`. Coordinates screened as text before casting. Parties per coordinate are
counted over the whole snapshot (not just customers): `PLACEHOLDER_MIN`+ is a placeholder (PNC towers, registered
agents, lockboxes, address vendors) unless it is a single post office (PO box geocode, one ZIP5); `COLOC_MIN`+ with
several ZIP5 on the point is a centroid fallback (placeholder); `COLOC_MIN`+ otherwise is `shared_building` (a house
or an office building: usable). APO / DPO / FPO states count as foreign. `geo_status` precedence: missing > foreign >
placeholder > pnc_address > reg_agent > po_box > low_precision > shared_building > ok. Self-scoring uses
`TRUTH_STATUSES` as targets (a PO box point is the post office, not the customer). `moved_12m` compares with the snapshot
`MOVE_LOOKBACK_DAYS` older (a move = a different ZIP5 and more than `MOVE_MIN_KM` apart). ZIP3 centres = median point
of usable customer addresses per ZIP3, with the ZIP3's main state: the cell of any estimated point is the nearest centre.

In [ ]:
# §2  Customer locations
t0 = time.time()
def addr_frame(snap):
    a = spark.table(T_ADDR).where(F.col("load_dt").cast("string") == snap)
    lat_s, lon_s = F.trim(F.col("latitude_degrees").cast("string")), F.trim(F.col("longitude_degrees").cast("string"))
    num = r"^-?\d{1,3}(\.\d+)?$"
    ok_s = lat_s.rlike(num) & lon_s.rlike(num)
    lat = F.when(ok_s, lat_s.cast("double")); lon = F.when(ok_s, lon_s.cast("double"))
    dec = lambda s: F.when(s.contains("."), F.length(F.substring_index(s, ".", -1))).otherwise(F.lit(0))
    zip_d = F.regexp_replace(F.coalesce(F.col("zip_cd").cast("string"), F.lit("")), r"[^0-9]", "")
    l1 = F.upper(F.coalesce(F.col("addr_line_1"), F.lit("")))
    w = Window.partitionBy("mdm_id").orderBy(F.col("addr_cleansed_date").desc_nulls_last(), F.col("mdm_address_id"))
    return (a.select(F.trim(F.col("mdm_id")).alias("mdm_id"), "mdm_address_id", "addr_loc_rec_type", "addr_cleansed_date",
                     lat.alias("lat"), lon.alias("lon"), F.least(dec(lat_s), dec(lon_s)).alias("coord_decimals"),
                     F.when(F.length(zip_d) >= 5, F.substring(zip_d, 1, 5)).alias("zip5"),
                     F.when(F.length(zip_d) >= 3, F.substring(zip_d, 1, 3)).alias("zip3"),
                     F.upper(F.trim(F.col("state_or_province"))).alias("state"),
                     F.upper(F.trim(F.col("city"))).alias("city"),
                     F.upper(F.trim(F.coalesce(F.col("addr_country"), F.lit("")))).alias("country"),
                     l1.rlike(PO_BOX_RE).alias("flag_po_box"), l1.rlike(REG_AGENT_RE).alias("flag_reg_agent"),
                     l1.rlike(CARE_OF_RE).alias("flag_care_of"), l1.rlike(PNC_RE).alias("flag_pnc"))
             .withColumn("_r", F.row_number().over(w)).where("_r = 1").drop("_r"))

A = addr_frame(SNAP)
A = A.withColumn("coord_key", F.when(F.col("lat").isNotNull(),
                                     F.concat_ws("_", F.round("lat", 5).cast("string"), F.round("lon", 5).cast("string"))))
coloc = (A.where(F.col("coord_key").isNotNull()).groupBy("coord_key")
           .agg(F.count("*").alias("coloc_cluster_size"), F.countDistinct("zip5").alias("coloc_n_zip5")))
A = A.join(coloc, "coord_key", "left")
dom = F.col("country").isin(US_COUNTRIES)
bad_xy = (F.col("lat").isNull() | (F.abs("lat") > 90) | (F.abs("lon") > 180) | ((F.col("lat") == 0) & (F.col("lon") == 0))
          | (dom & (F.col("lon") > 0)))                       # positive longitude inside the US = lost minus sign
is_po = F.col("flag_po_box") | F.coalesce(F.col("addr_loc_rec_type").isin(PO_REC_TYPES), F.lit(False))
size_, nz_ = F.coalesce(F.col("coloc_cluster_size"), F.lit(0)), F.coalesce(F.col("coloc_n_zip5"), F.lit(0))
is_ph = (((size_ >= PLACEHOLDER_MIN) & ~(is_po & (nz_ <= 1)))          # a big point that is not a single post office
         | ((size_ >= COLOC_MIN) & (nz_ > 1)))                         # one point carrying several ZIP5: centroid fallback
A = A.withColumn("geo_status",
                 F.when(bad_xy & dom, "missing")
                  .when(~dom | F.coalesce(F.col("state").isin(MILITARY_STATES), F.lit(False)), "foreign")
                  .when(is_ph, "placeholder")
                  .when(F.col("flag_pnc"), "pnc_address").when(F.col("flag_reg_agent"), "reg_agent")
                  .when(is_po, "po_box").when(F.col("coord_decimals") <= LOW_PRECISION_DP, "low_precision")
                  .when(size_ >= COLOC_MIN, "shared_building")
                  .otherwise("ok"))
A = A.withColumn("geo_ok", F.col("geo_status").isin(GEO_OK_STATUSES))
if SNAP_PREV:
    P = addr_frame(SNAP_PREV).select("mdm_id", F.col("lat").alias("p_lat"), F.col("lon").alias("p_lon"),
                                     F.col("zip5").alias("p_zip5"), F.col("zip3").alias("prev_zip3"))
    A = A.join(P, "mdm_id", "left")
    x1, y1, z1 = unit(F.col("lat"), F.col("lon")); x2, y2, z2 = unit(F.col("p_lat"), F.col("p_lon"))
    A = (A.withColumn("move_km", F.when(F.col("p_lat").isNotNull() & F.col("lat").isNotNull(), chord_km(x1, y1, z1, x2, y2, z2)))
          .withColumn("moved_12m", F.when(F.col("p_lat").isNotNull() | F.col("p_zip5").isNotNull(),
                                          (F.coalesce(F.col("p_zip5"), F.lit("")) != F.coalesce(F.col("zip5"), F.lit("")))
                                          & (F.coalesce(F.col("move_km"), F.lit(1e9)) > MOVE_MIN_KM)))
          .drop("p_lat", "p_lon", "p_zip5"))
else:
    A = (A.withColumn("prev_zip3", F.lit(None).cast("string")).withColumn("move_km", F.lit(None).cast("double"))
          .withColumn("moved_12m", F.lit(None).cast("boolean")))

cust = spark.table(T_CUST).select(F.trim(F.col("mdm_id")).alias("mdm_id"), "party_type",
                                  F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)).alias("is_tm"), "naics2")
G0 = (cust.join(A.drop("coord_key"), "mdm_id", "left")
          .withColumn("geo_status", F.coalesce(F.col("geo_status"), F.lit("missing")))
          .withColumn("geo_ok", F.coalesce(F.col("geo_ok"), F.lit(False)))
          .withColumn("snapshot_dt", F.lit(SNAP)))
x, y, z = unit(F.col("lat"), F.col("lon"))
G0 = G0.withColumn("ux", F.when(F.col("geo_ok"), x)).withColumn("uy", F.when(F.col("geo_ok"), y)).withColumn("uz", F.when(F.col("geo_ok"), z))
G0.write.mode("overwrite").parquet(f"{WORK}/g0")
G0 = spark.read.parquet(f"{WORK}/g0")
GEO = G0.where("geo_ok").select("mdm_id", "lat", "lon", "ux", "uy", "uz", "zip3", "state", "geo_status")   # located customers
GEO.write.mode("overwrite").parquet(f"{WORK}/geo"); GEO = spark.read.parquet(f"{WORK}/geo")

# ZIP3 centres and the cell lookup (grid of 1° cells, each centre replicated to its 3×3 neighbourhood)
Z3 = (GEO.where(F.col("zip3").isNotNull())
         .groupBy("zip3").agg(F.expr("percentile_approx(lat, 0.5)").alias("zlat"), F.expr("percentile_approx(lon, 0.5)").alias("zlon"),
                              F.count("*").alias("n_customers")))
Z3S = (GEO.where(F.col("zip3").isNotNull() & F.col("state").isNotNull()).groupBy("zip3", "state").count()
          .withColumn("_r", F.row_number().over(Window.partitionBy("zip3").orderBy(F.desc("count"), "state")))
          .where("_r = 1").select("zip3", F.col("state").alias("zstate")))
Z3 = Z3.join(Z3S, "zip3", "left").toPandas()
Z3.to_csv(f"{OUT_DIR}/zip3_centres.csv", index=False)
_zr = []
for _, r in Z3.iterrows():
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            _zr.append((int(math.floor(r.zlat)) + di, int(math.floor(r.zlon)) + dj, r.zip3, float(r.zlat), float(r.zlon), r.zstate))
ZGRID_PD = pd.DataFrame(_zr, columns=["gi", "gj", "cell_zip3", "zlat", "zlon", "cell_state"])
def zgrid():
    """The cell lookup as a DataFrame of the current session (rebuilt on demand: §3 may start new sessions)."""
    return F.broadcast(spark.createDataFrame(ZGRID_PD)) if len(ZGRID_PD) else None

def add_cell(df, lat_c, lon_c, prefix="cell"):
    """Nearest ZIP3 centre (within the 3×3 one-degree neighbourhood) → <prefix>_zip3, <prefix>_state."""
    ZG = zgrid()
    if ZG is None:
        return df.withColumn(f"{prefix}_zip3", F.lit(None).cast("string")).withColumn(f"{prefix}_state", F.lit(None).cast("string"))
    k = df.select(F.col(lat_c).alias("_la"), F.col(lon_c).alias("_lo")).where(F.col("_la").isNotNull()).distinct()
    k = k.withColumn("gi", F.floor("_la").cast("int")).withColumn("gj", F.floor("_lo").cast("int"))
    j = (k.join(ZG, ["gi", "gj"])
          .withColumn("_d", (F.col("_la") - F.col("zlat")) ** 2 + ((F.col("_lo") - F.col("zlon")) * F.cos(F.radians("_la"))) ** 2)
          .withColumn("_r", F.row_number().over(Window.partitionBy("_la", "_lo").orderBy("_d", "cell_zip3")))
          .where("_r = 1").select("_la", "_lo", F.col("cell_zip3").alias(f"{prefix}_zip3"), F.col("cell_state").alias(f"{prefix}_state")))
    return (df.join(j, (df[lat_c].eqNullSafe(j["_la"])) & (df[lon_c].eqNullSafe(j["_lo"])), "left").drop("_la", "_lo"))

st = (G0.groupBy("geo_status", F.when(F.col("party_type") == "O", "organisation").otherwise("person / other").alias("party"))
        .agg(F.count("*").alias("customers"), F.sum(F.col("is_tm").cast("int")).alias("tm_customers")).toPandas()
        .sort_values(["party", "customers"], ascending=[True, False]))
mv = G0.agg(F.sum(F.col("moved_12m").cast("int")).alias("moved_12m"), F.sum(F.col("geo_ok").cast("int")).alias("geo_ok"),
            F.count("*").alias("customers")).first()
out(sec(f"§2 Customer locations · snapshot {SNAP} · {mv['customers']:,} customers, {mv['geo_ok']:,} usable "
        f"({mv['geo_ok'] / max(mv['customers'], 1):.1%}) · moved in 12 months {mv['moved_12m'] or 0:,} · "
        f"{len(Z3):,} ZIP3 centres · {(time.time() - t0) / 60:.1f} min"),
    tbl(st))

## §3 Monthly block (`pkg_cust_geo_monthly`)

Per month, every customer's counterparty dollars by direction from `pkg_edge_monthly` (own accounts left out). A
counterparty is **located** only when it is a PNC customer with a usable address. Dollar-weighted, on 3D unit
vectors: centroid and resultant length `geo_R`; spread = radius of gyration about the centroid (NULL below
`MIN_CP_SPREAD` located counterparties); ZIP3 breadth (`geo_n_zip3_80`) and entropy (NULL below `MIN_CP_ENTROPY`);
home ZIP3 / state shares of located dollars (0 when there is something to compare and nothing matches); reach from
the customer's own point (NULL without one, never 0); coverage = located share of all dollars of that direction.
Every column twice: all counterparties, and `_xh` without hubs (degree ≥ `HUB_MULT` × the median this month).
Reconciles exactly to the edges every month. One month at a time, a fresh Spark session after each.

In [ ]:
# §3  Monthly block
LOG = []
_PH = display(HTML(""), display_id=True)
_CUR = {"m": None, "step": "", "t": None}
_PLOCK = threading.Lock()
def _panel(final=None):
    with _PLOCK:
        _panel_(final)
def _panel_(final=None):
    lg = pd.DataFrame(LOG)
    n_done = len(lg)
    mpm = lg["minutes"].mean() if n_done else None
    left = (len(TODO) - n_done) * mpm if mpm else None
    head = (f"§3 {n_done} of {len(TODO)} month(s) built this run"
            + (f" · {mpm:.1f} min per month · about {left / 60:.1f} h left" if left is not None else ""))
    now = (f"<br>now: <b>{_CUR['m']}</b> · {_CUR['step']} · {(time.time() - _CUR['t']) / 60:.1f} min into the month"
           if _CUR["m"] and final is None else "")
    _PH.update(HTML("<div>" + (final or "") + f"<p style='margin:4px 0'>{head}{now}<br><span style='font-size:12px'>"
                    f"updated {dt.datetime.now().strftime('%H:%M:%S')}</span></p>" + (tbl(lg) if n_done else "") + "</div>"))
_HB = threading.Event()
def _heartbeat():
    while not _HB.wait(30):
        try:
            if _CUR["m"]:
                _panel()
        except Exception:
            pass
_HBT = threading.Thread(target=_heartbeat, daemon=True); _HBT.start()
def _stop_heartbeat():
    _CUR["m"] = None; _HB.set(); _HBT.join(timeout=60)
def step(s):
    _CUR["step"] = s; _panel()

def side_rows(m):
    """Customer-side rows of month m: cust, dir, other, okind, amt (own accounts left out; duplicates summed)."""
    e = spark.table(T_EDGE_M).where((F.col("month") == m) & ~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
    o = e.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.lit("out").alias("dir"),
                                                    F.col("dst_id").alias("other"), F.col("dst_kind").alias("okind"), "amount")
    i = e.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.lit("in").alias("dir"),
                                                    F.col("src_id").alias("other"), F.col("src_kind").alias("okind"), "amount")
    return (o.unionByName(i).groupBy("cust", "dir", "other", "okind")
             .agg(F.sum("amount").alias("amt_dec")).withColumn("amt", F.col("amt_dec").cast("double")))

def edge_side_total(m):
    e = spark.table(T_EDGE_M).where((F.col("month") == m) & ~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
    return e.agg(F.sum(F.when(F.col("src_kind") == "CUST", F.col("amount")).otherwise(0)).alias("o"),
                 F.sum(F.when(F.col("dst_kind") == "CUST", F.col("amount")).otherwise(0)).alias("i")).first()

def block(L, own, sfx):
    """Block F on located rows L (cust, dir, other, amt, ux, uy, uz, zip3, state) for one hub rung; own = the
    customer's own point and ZIP3 / state. Returns one row per customer with the 21 non-coverage columns + sfx."""
    LS = L.withColumn("scope", F.explode(F.array(F.col("dir"), F.lit("all"))))
    c = (LS.groupBy("cust", "scope")
           .agg(F.sum("amt").alias("S"), F.sum(F.col("amt") * F.col("ux")).alias("vx"), F.sum(F.col("amt") * F.col("uy")).alias("vy"),
                F.sum(F.col("amt") * F.col("uz")).alias("vz"), F.countDistinct("other").alias("n"))
           .withColumn("vn", F.sqrt(F.col("vx") ** 2 + F.col("vy") ** 2 + F.col("vz") ** 2))
           .withColumn("R", F.when(F.col("S") > 0, F.col("vn") / F.col("S")))
           .withColumn("cx", F.col("vx") / F.col("vn")).withColumn("cy", F.col("vy") / F.col("vn")).withColumn("cz", F.col("vz") / F.col("vn")))
    sp = (LS.join(c.select("cust", "scope", "cx", "cy", "cz", "S"), ["cust", "scope"])
            .withColumn("d", chord_km(F.col("ux"), F.col("uy"), F.col("uz"), F.col("cx"), F.col("cy"), F.col("cz")))
            .groupBy("cust", "scope").agg((F.sum(F.col("amt") * F.col("d") ** 2) / F.first("S")).alias("g2")))
    c = c.join(sp, ["cust", "scope"], "left").withColumn("spread", F.when(F.col("n") >= MIN_CP_SPREAD, F.sqrt("g2")))
    clat, clon = latlon(F.col("cx"), F.col("cy"), F.col("cz"))
    allc = c.where("scope = 'all'").select("cust", F.col("spread").alias("geo_spread_km"), clat.alias("geo_centroid_lat"),
                                           clon.alias("geo_centroid_lon"), F.col("R").alias("geo_R"), F.col("n").alias("_n"),
                                           "cx", "cy", "cz")
    din = c.where("scope = 'in'").select("cust", F.col("spread").alias("geo_spread_in_km"))
    dout = c.where("scope = 'out'").select("cust", F.col("spread").alias("geo_spread_out_km"))
    # ZIP3 breadth and entropy (both directions pooled)
    zs = (L.where(F.col("zip3").isNotNull()).groupBy("cust", "zip3").agg(F.sum("amt").alias("a"))
           .withColumn("p", F.col("a") / F.sum("a").over(Window.partitionBy("cust")))
           .withColumn("cum", F.sum("p").over(Window.partitionBy("cust").orderBy(F.desc("p"), "zip3")
                                              .rowsBetween(Window.unboundedPreceding, 0)))
           .groupBy("cust").agg(F.sum(F.when(F.col("cum") - F.col("p") < 0.8, 1).otherwise(0)).alias("geo_n_zip3_80"),
                                F.greatest(F.lit(0.0), -F.sum(F.col("p") * F.log("p"))).alias("_ent")))
    # home shares and reach need the customer's own point
    Lo = L.join(own, "cust")
    hs = (Lo.groupBy("cust").pivot("dir", ["in", "out"])
            .agg(F.sum("amt").alias("tot"),
                 F.sum(F.when(F.col("zip3") == F.col("own_zip3"), F.col("amt")).otherwise(0)).alias("hz"),
                 F.sum(F.when(F.col("state") == F.col("own_state"), F.col("amt")).otherwise(0)).alias("hst")))
    hs = hs.select("cust", *[F.when(F.col(f"{d}_tot") > 0, F.col(f"{d}_{a}") / F.col(f"{d}_tot")).alias(f"geo_home_{k}_share_{d}")
                             for d in ("in", "out") for k, a in (("zip3", "hz"), ("state", "hst"))])
    rd = (Lo.withColumn("d", chord_km(F.col("ux"), F.col("uy"), F.col("uz"), F.col("own_x"), F.col("own_y"), F.col("own_z")))
            .withColumn("cw", F.sum("amt").over(Window.partitionBy("cust").orderBy("d", "other", "dir")
                                                .rowsBetween(Window.unboundedPreceding, 0)))
            .withColumn("W", F.sum("amt").over(Window.partitionBy("cust")))
            .groupBy("cust").agg((F.sum(F.col("amt") * F.col("d")) / F.first("W")).alias("geo_reach_mean_km"),
                                 F.min(F.when(F.col("cw") >= 0.5 * F.col("W"), F.col("d"))).alias("geo_reach_p50_km"),
                                 F.min(F.when(F.col("cw") >= 0.9 * F.col("W"), F.col("d"))).alias("geo_reach_p90_km")))
    r = (allc.join(din, "cust", "left").join(dout, "cust", "left").join(zs, "cust", "left").join(hs, "cust", "left")
             .join(rd, "cust", "left").join(own.select("cust", "own_x", "own_y", "own_z"), "cust", "left")
             .withColumn("geo_zip3_entropy", F.when(F.col("_n") >= MIN_CP_ENTROPY, F.col("_ent")))
             .withColumn("geo_registered_vs_flow_km",
                         F.when(F.col("own_x").isNotNull(), chord_km(F.col("cx"), F.col("cy"), F.col("cz"), F.col("own_x"), F.col("own_y"), F.col("own_z"))))
             .withColumn("geo_locality_class", F.when(F.col("geo_spread_km") < LOCALITY_KM[0], "LOCAL")
                         .when(F.col("geo_spread_km") < LOCALITY_KM[1], "REGIONAL").when(F.col("geo_spread_km") < LOCALITY_KM[2], "MULTI_MARKET")
                         .when(F.col("geo_spread_km").isNotNull(), "NATIONAL"))
             .drop("_n", "_ent", "cx", "cy", "cz", "own_x", "own_y", "own_z"))
    return r.select("cust", *[F.col(c_).alias(c_ + sfx) for c_ in r.columns if c_ != "cust"])

BLOCK_COLS = ["geo_spread_km", "geo_spread_in_km", "geo_spread_out_km", "geo_centroid_lat", "geo_centroid_lon", "geo_R",
              "geo_n_zip3_80", "geo_zip3_entropy", "geo_locality_class", "geo_home_zip3_share_in", "geo_home_zip3_share_out",
              "geo_home_state_share_in", "geo_home_state_share_out", "geo_reach_mean_km", "geo_reach_p50_km",
              "geo_reach_p90_km", "geo_registered_vs_flow_km", "geo_cov_amt_in", "geo_cov_amt_out",
              "geo_n_cp_located_in", "geo_n_cp_located_out"]

def build_month(m):
    t_ = time.time()
    _CUR.update(m=m, t=time.time())
    step("1/3 counterparty dollars and hubs")
    geo = spark.read.parquet(f"{WORK}/geo")
    S_ = side_rows(m)
    S_.write.mode("overwrite").parquet(f"{WORK}/m_rows"); S_ = spark.read.parquet(f"{WORK}/m_rows")
    deg = S_.groupBy("other").agg(F.countDistinct("cust").alias("deg"))
    med = deg.approxQuantile("deg", [0.5], 0.001)[0] or 1.0
    hub_cut = HUB_MULT * med
    hubs = deg.where(F.col("deg") >= hub_cut).select("other", F.lit(True).alias("is_hub"))
    n_hubs = hubs.count()
    L = (S_.where(F.col("okind") == "CUST").join(geo.select(F.col("mdm_id").alias("other"), "ux", "uy", "uz", "zip3", "state"), "other")
           .join(F.broadcast(hubs) if n_hubs < 200000 else hubs, "other", "left")
           .withColumn("is_hub", F.coalesce("is_hub", F.lit(False))))
    L.write.mode("overwrite").parquet(f"{WORK}/m_loc"); L = spark.read.parquet(f"{WORK}/m_loc")
    own = geo.select(F.col("mdm_id").alias("cust"), F.col("ux").alias("own_x"), F.col("uy").alias("own_y"),
                     F.col("uz").alias("own_z"), F.col("zip3").alias("own_zip3"), F.col("state").alias("own_state"))
    step("2/3 Block F, all counterparties and without hubs")
    tot = (S_.groupBy("cust").pivot("dir", ["in", "out"]).agg(F.sum("amt"))
             .withColumnRenamed("in", "in_T").withColumnRenamed("out", "out_T"))
    loc_ = (L.groupBy("cust").pivot("dir", ["in", "out"]).agg(F.sum("amt").alias("A"), F.countDistinct("other").alias("N")))
    locx = (L.where(~F.col("is_hub")).groupBy("cust").pivot("dir", ["in", "out"]).agg(F.sum("amt").alias("A"), F.countDistinct("other").alias("N")))
    def cov(df, sfx):
        return df.select("cust", *[F.col(f"{d}_A").alias(f"_a_{d}") for d in ("in", "out")],
                         *[F.col(f"{d}_N").cast("long").alias(f"geo_n_cp_located_{d}{sfx}") for d in ("in", "out")])
    R_ = (tot.join(cov(loc_, ""), "cust", "left").join(cov(locx, "_xh").withColumnRenamed("_a_in", "_ax_in").withColumnRenamed("_a_out", "_ax_out"), "cust", "left")
             .join(block(L, own, ""), "cust", "left").join(block(L.where(~F.col("is_hub")), own, "_xh"), "cust", "left"))
    for d in ("in", "out"):
        R_ = (R_.withColumn(f"geo_cov_amt_{d}", F.when(F.col(f"{d}_T") > 0, F.coalesce(F.col(f"_a_{d}"), F.lit(0.0)) / F.col(f"{d}_T")))
                .withColumn(f"geo_cov_amt_{d}_xh", F.when(F.col(f"{d}_T") > 0, F.coalesce(F.col(f"_ax_{d}"), F.lit(0.0)) / F.col(f"{d}_T")))
                .withColumn(f"geo_n_cp_located_{d}", F.coalesce(F.col(f"geo_n_cp_located_{d}"), F.lit(0).cast("long")))
                .withColumn(f"geo_n_cp_located_{d}_xh", F.coalesce(F.col(f"geo_n_cp_located_{d}_xh"), F.lit(0).cast("long"))))
    # home shares: 0 (not NULL) when there are located dollars and an own key; NULL without own location
    R_ = R_.join(own.select("cust", "own_zip3", "own_state"), "cust", "left")
    for s_ in ("", "_xh"):
        for d in ("in", "out"):
            for k in ("zip3", "state"):
                c_ = f"geo_home_{k}_share_{d}{s_}"
                R_ = R_.withColumn(c_, F.when(F.col(f"own_{k}").isNull(), F.lit(None).cast("double"))
                                    .when(F.col(f"geo_n_cp_located_{d}{s_}") > 0, F.coalesce(F.col(c_), F.lit(0.0))))
    keep = ["cust"] + BLOCK_COLS + [c_ + "_xh" for c_ in BLOCK_COLS]
    if MONTHLY_TM_ORG_ONLY:
        R_ = R_.join(spark.table(T_CUST).where(F.coalesce(F.col("is_tm").cast("boolean"), F.lit(False)) | (F.col("party_type") == "O"))
                        .select(F.trim(F.col("mdm_id")).alias("cust")).distinct(), "cust", "left_semi")
    step("3/3 write and reconcile")
    p = f"{LOC['monthly']}/month={m}"
    delete(p)
    R_.select(*keep).withColumnRenamed("cust", "mdm_id").write.mode("overwrite").parquet(p)
    exp = edge_side_total(m)
    s_dec = spark.read.parquet(f"{WORK}/m_rows").groupBy("dir").agg(F.sum("amt_dec").alias("a")).toPandas().set_index("dir")["a"].to_dict()
    ok = (s_dec.get("out", 0) == (exp["o"] or 0)) and (s_dec.get("in", 0) == (exp["i"] or 0))
    assert ok, f"{m}: customer-side dollars {s_dec} ≠ edges out {exp['o']} / in {exp['i']}"
    n = spark.read.parquet(p).count()
    write_json(f"{p}/_GEO_BUILD", {"sig": SIG, "snapshot": SNAP, "built_at": dt.datetime.now().isoformat(timespec="seconds"),
                                   "rows": n, "hub_cut": hub_cut, "n_hubs": n_hubs})
    return {"month": m, "rows": n, "hubs": n_hubs, "hub_degree_cut": round(hub_cut, 1),
            "usd_bn_out": float(exp["o"] or 0) / 1e9, "usd_bn_in": float(exp["i"] or 0) / 1e9, "reconciled": "yes",
            "minutes": round((time.time() - t_) / 60, 2)}

def register_monthly():
    k, t = "monthly", OUT_TABLES["monthly"]
    parts_ = sorted(x for x in ls_dirs(LOC[k]) if x.startswith("month="))
    if not parts_:
        return
    sch = spark.read.parquet(f"{LOC[k]}/{parts_[-1]}").schema
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
    if table_exists(t) and [c for c in spark.table(t).columns if c != "month"] != [f.name for f in sch.fields]:
        spark.sql(f"DROP TABLE {t}")
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {t} ({cols}) PARTITIONED BY (month string) STORED AS PARQUET LOCATION '{LOC[k]}'")
    spark.sql(f"ALTER TABLE {t} RECOVER PARTITIONS")
    spark.sql(f"REFRESH TABLE {t}")

_panel()
for i_, m in enumerate(TODO):
    try:
        LOG.append(build_month(m))
    except Exception as e:
        _stop_heartbeat()
        _panel(sec(f"Stopped at {m}: {str(e).splitlines()[0][:200]}. Finished months are kept: rerun this cell (or Run All) "
                   f"and it resumes at {m}."))
        raise
    pd.DataFrame([{**LOG[-1], "finished_at": dt.datetime.now().isoformat(timespec="seconds")}]).to_csv(
        f"{OUT_DIR}/run_log.csv", mode="a", header=not os.path.exists(f"{OUT_DIR}/run_log.csv"), index=False)
    _CUR["m"] = None; _panel()
    if NEW_SESSION_EACH_MONTH and i_ < len(TODO) - 1:
        _CUR.update(m=TODO[i_ + 1], step="starting a fresh Spark session", t=time.time()); _panel()
        new_session()
for _x in ("m_rows", "m_loc"):
    delete(f"{WORK}/{_x}")
register_monthly()
_stop_heartbeat()
_panel(sec("§3 Done" if LOG else "§3 Nothing to build: every month in scope is done for this signature"))

## §4 Neighbour kernels, self-scoring and calibration

**Kernel** of customer j over the last `KERNEL_MONTHS` months: the unweighted sum of its distinct located
counterparties' unit vectors (V, n). Centre = V / |V|, width = R_earth · sqrt(2 (1 − R̄)), R̄ = |V| / n. With a target t
inside j's cloud, leave-one-out is a subtraction (V − u_t, n − 1). Width NULL below `MIN_CP_BW` counterparties after
leave-one-out, not used below `MIN_CP_BW_TRUST`, and a kernel narrower than `COLOC_WIDTH_KM` is a co-located cluster:
not used.

**Self-scoring:** every located customer t is predicted from its neighbours j (customers it pays or is paid by in the
window) with itself left out of each kernel, then compared with its address. The usable neighbours are ranked by width
(tightest first) and the first `K_MAX` combined with weights 1 / width²; predicted width = 1 / sqrt(Σ 1 / width²).
Half of the targets (hash) fit the **radius calibration** (realised error quantiles per predicted-width bin); the other
half checks it. The same scores give the error-vs-k curve (tightest-first vs random order, fixed cohort with ≥ `K_MAX`
neighbours), the gate, the footprint-prior baseline, truth sensitivity, footprint bias and each neighbour's locating
class (median single-neighbour error).

In [ ]:
# §4  Kernels, self-scoring, calibration
t0 = time.time()
GEO = spark.read.parquet(f"{WORK}/geo")
def window_pairs():
    """Distinct customer–counterparty pairs of the kernel window (both directions, own accounts left out)."""
    e = spark.table(T_EDGE_M).where(F.col("month").isin(K_MONTHS) & ~F.coalesce(F.col("is_self").cast("boolean"), F.lit(False)))
    a = e.where(F.col("src_kind") == "CUST").select(F.col("src_id").alias("cust"), F.col("dst_id").alias("other"),
                                                    F.col("dst_kind").alias("okind"), F.col("amount").cast("double").alias("amt"))
    b = e.where(F.col("dst_kind") == "CUST").select(F.col("dst_id").alias("cust"), F.col("src_id").alias("other"),
                                                    F.col("src_kind").alias("okind"), F.col("amount").cast("double").alias("amt"))
    return a.unionByName(b).groupBy("cust", "other", "okind").agg(F.sum("amt").alias("amt"))
WP = window_pairs()
WP.write.mode("overwrite").parquet(f"{WORK}/wp"); WP = spark.read.parquet(f"{WORK}/wp")

# kernels (unweighted) + the dollar-weighted flow centre (registered vs flow) + degree / hub over the window
g = GEO.select(F.col("mdm_id").alias("other"), "ux", "uy", "uz")
KL = WP.where(F.col("okind") == "CUST").join(g, "other")
K = (KL.groupBy("cust").agg(F.sum("ux").alias("k_sx"), F.sum("uy").alias("k_sy"), F.sum("uz").alias("k_sz"),
                            F.count("*").alias("k_n_cp"),
                            F.sum(F.col("amt") * F.col("ux")).alias("_wx"), F.sum(F.col("amt") * F.col("uy")).alias("_wy"),
                            F.sum(F.col("amt") * F.col("uz")).alias("_wz"))
       .withColumn("k_sw", F.col("k_n_cp").cast("double")))
deg = WP.groupBy("other").agg(F.countDistinct("cust").alias("degree"))
med = deg.approxQuantile("degree", [0.5], 0.001)[0] or 1.0
HUB_CUT = HUB_MULT * med
K = (K.join(deg.withColumnRenamed("other", "cust"), "cust", "left")
      .withColumn("is_hub", F.coalesce(F.col("degree"), F.lit(0)) >= HUB_CUT)
      .withColumn("_rn", F.sqrt(F.col("k_sx") ** 2 + F.col("k_sy") ** 2 + F.col("k_sz") ** 2))
      .withColumn("k_width_km", F.when(F.col("k_n_cp") >= MIN_CP_BW, width_km(F.col("_rn") / F.col("k_n_cp")))))
K.write.mode("overwrite").parquet(f"{WORK}/k"); K = spark.read.parquet(f"{WORK}/k")

def usable(df):
    """Neighbour kernels after leave-one-out (columns kx, ky, kz, kn): centre and width, the guards."""
    rn = F.sqrt(F.col("kx") ** 2 + F.col("ky") ** 2 + F.col("kz") ** 2)
    return (df.withColumn("_rn", rn)
              .withColumn("w", F.when((F.col("kn") >= MIN_CP_BW) & (F.col("_rn") > 0), width_km(F.col("_rn") / F.col("kn"))))
              .withColumn("cx", F.col("kx") / F.col("_rn")).withColumn("cy", F.col("ky") / F.col("_rn")).withColumn("cz", F.col("kz") / F.col("_rn"))
              .where((F.col("kn") >= MIN_CP_BW_TRUST) & F.col("w").isNotNull() & (F.col("w") >= COLOC_WIDTH_KM))
              .drop("_rn"))

def combine(N, k_max=K_MAX, order="w"):
    """N: target, j, cx, cy, cz, w → per target: estimate (tightest k_max, inverse-variance), predicted width."""
    # exact top k_max in two stages (per target × hash bucket, then per target): a target with millions of neighbours
    # (a national payee) never sorts in one task
    r0 = (N.withColumn("_s", F.abs(F.xxhash64("j")) % 64)
           .withColumn("_rk0", F.row_number().over(Window.partitionBy("target", "_s").orderBy(order, "j")))
           .where(F.col("_rk0") <= k_max).drop("_s", "_rk0"))
    r = (r0.withColumn("_rk", F.row_number().over(Window.partitionBy("target").orderBy(order, "j")))
           .where(F.col("_rk") <= k_max).withColumn("iv", 1 / (F.col("w") ** 2)))
    e = (r.groupBy("target").agg(F.sum(F.col("iv") * F.col("cx")).alias("ex"), F.sum(F.col("iv") * F.col("cy")).alias("ey"),
                                 F.sum(F.col("iv") * F.col("cz")).alias("ez"), F.sum("iv").alias("siv"),
                                 F.count("*").alias("k_used"), F.min("w").alias("tightest_bw_km"))
          .withColumn("_n", F.sqrt(F.col("ex") ** 2 + F.col("ey") ** 2 + F.col("ez") ** 2))
          .withColumn("ex", F.col("ex") / F.col("_n")).withColumn("ey", F.col("ey") / F.col("_n")).withColumn("ez", F.col("ez") / F.col("_n"))
          .withColumn("pred_width_km", 1 / F.sqrt("siv")).drop("_n", "siv"))
    la, lo = latlon(F.col("ex"), F.col("ey"), F.col("ez"))
    return e.withColumn("est_lat", la).withColumn("est_lon", lo)

# self-scoring pairs: target t (located customer) – neighbour j (customer with a kernel), t left out of j's kernel
tgt = (GEO.where(F.col("geo_status").isin(TRUTH_STATUSES))
          .select(F.col("mdm_id").alias("target"), F.col("ux").alias("tx"), F.col("uy").alias("ty"), F.col("uz").alias("tz"),
                  "state", "geo_status"))
if SCORE_PCT < 100:
    tgt = tgt.where(F.abs(F.xxhash64("target")) % 100 < SCORE_PCT)
SP = (WP.where(F.col("okind") == "CUST").select(F.col("other").alias("target"), F.col("cust").alias("j")).distinct()
        .join(tgt, "target")
        .join(K.select(F.col("cust").alias("j"), "k_sx", "k_sy", "k_sz", "k_n_cp"), "j")
        .select("target", "j", "tx", "ty", "tz", "state", (F.col("k_sx") - F.col("tx")).alias("kx"), (F.col("k_sy") - F.col("ty")).alias("ky"),
                (F.col("k_sz") - F.col("tz")).alias("kz"), (F.col("k_n_cp") - 1).alias("kn")))
SP = usable(SP).withColumn("err1", chord_km(F.col("cx"), F.col("cy"), F.col("cz"), F.col("tx"), F.col("ty"), F.col("tz")))
SP.write.mode("overwrite").parquet(f"{WORK}/sp"); SP = spark.read.parquet(f"{WORK}/sp")

# registered vs flow and representativeness, for every customer with a kernel
rvf = (K.join(GEO.select(F.col("mdm_id").alias("cust"), F.col("ux").alias("ox"), F.col("uy").alias("oy"), F.col("uz").alias("oz")), "cust")
        .withColumn("_wn", F.sqrt(F.col("_wx") ** 2 + F.col("_wy") ** 2 + F.col("_wz") ** 2))
        .where(F.col("_wn") > 0)
        .select("cust", chord_km(F.col("_wx") / F.col("_wn"), F.col("_wy") / F.col("_wn"), F.col("_wz") / F.col("_wn"),
                                 F.col("ox"), F.col("oy"), F.col("oz")).alias("geo_registered_vs_flow_km")))
rvf.write.mode("overwrite").parquet(f"{WORK}/rvf"); rvf = spark.read.parquet(f"{WORK}/rvf")

# combined estimates and errors per target; calibration on the even hash half, check on the odd half
EST = (combine(SP.select("target", "j", "cx", "cy", "cz", "w"))
       .join(tgt.select("target", "tx", "ty", "tz", "state", "geo_status"), "target")
       .withColumn("err", chord_km(F.col("ex"), F.col("ey"), F.col("ez"), F.col("tx"), F.col("ty"), F.col("tz")))
       .join(rvf.withColumnRenamed("cust", "target"), "target", "left")
       .withColumn("pin_representative", F.col("geo_registered_vs_flow_km") <= REP_GAP_KM)
       .withColumn("half", F.abs(F.xxhash64(F.concat(F.col("target"), F.lit(str(SEED))))) % 2))
EST.write.mode("overwrite").parquet(f"{WORK}/est"); EST = spark.read.parquet(f"{WORK}/est")

qs = [i / CALIB_BINS for i in range(1, CALIB_BINS)]
edges_ = sorted(set(EST.where("half = 0").approxQuantile("pred_width_km", qs, 0.001)))
_b_expr = F.lit(0)
for i, v in enumerate(edges_):
    _b_expr = F.when(F.col("pred_width_km") > v, F.lit(i + 1)).otherwise(_b_expr)
EST = EST.withColumn("wbin", _b_expr)
CAL = (EST.where("half = 0").groupBy("wbin")
          .agg(F.count("*").alias("n"), F.min("pred_width_km").alias("width_from_km"), F.max("pred_width_km").alias("width_to_km"),
               F.expr("percentile_approx(err, 0.5)").alias("r50_km"), F.expr("percentile_approx(err, 0.9)").alias("r90_km"),
               F.avg((F.col("err") <= 50).cast("double")).alias("p_within_50km"),
               F.avg((F.col("err") <= 250).cast("double")).alias("p_within_250km"))
          .orderBy("wbin").toPandas())
# r50 / r90 must not fall as the predicted width grows (monotone, so a wider estimate never claims a tighter radius)
for c_ in ("r50_km", "r90_km"):
    CAL[c_] = CAL[c_].cummax()
for c_ in ("p_within_50km", "p_within_250km"):
    CAL[c_] = CAL[c_][::-1].cummax()[::-1]
CAL["evidence_class"] = np.select([CAL.r50_km <= CLASS_KM["PIN"], CAL.r50_km <= CLASS_KM["METRO"], CAL.r50_km <= CLASS_KM["REGION"]],
                                  ["PIN", "METRO", "REGION"], "NONE")
CAL.to_csv(f"{OUT_DIR}/radius_calibration.csv", index=False)
CALDF = F.broadcast(spark.createDataFrame(CAL[["wbin", "r50_km", "r90_km", "p_within_50km", "p_within_250km", "evidence_class"]]))
def calibrate(E):
    b = F.lit(0)
    for i, v in enumerate(edges_):
        b = F.when(F.col("pred_width_km") > v, F.lit(i + 1)).otherwise(b)
    return E.withColumn("wbin", b).join(CALDF, "wbin", "left").drop("wbin")

chk = (calibrate(EST.drop("wbin")).where("half = 1")
          .agg(F.avg((F.col("err") <= F.col("r50_km")).cast("double")).alias("in_r50"),
               F.avg((F.col("err") <= F.col("r90_km")).cast("double")).alias("in_r90"),
               F.count("*").alias("n")).first())

# footprint prior: centre of all located customers (the baseline every estimate must beat)
fp = GEO.agg(F.sum("ux").alias("x"), F.sum("uy").alias("y"), F.sum("uz").alias("z")).first()
_fn = math.sqrt(fp.x ** 2 + fp.y ** 2 + fp.z ** 2)
EST = EST.withColumn("err_prior", chord_km(F.lit(fp.x / _fn), F.lit(fp.y / _fn), F.lit(fp.z / _fn), F.col("tx"), F.col("ty"), F.col("tz")))

# core states: the fewest states holding CORE_SHARE of located customers
sc = GEO.where(F.col("state").isNotNull()).groupBy("state").count().toPandas().sort_values("count", ascending=False)
sc["cum"] = sc["count"].cumsum() / sc["count"].sum()
CORE = sc.loc[sc["cum"].shift(fill_value=0) < CORE_SHARE, "state"].tolist()

def acc(df, by=None):
    aggs = [F.count("*").alias("targets"), F.expr("percentile_approx(err, 0.5)").alias("median_km"),
            F.expr("percentile_approx(err, 0.9)").alias("p90_km"),
            F.avg((F.col("err") <= 25).cast("double")).alias("hit_25"), F.avg((F.col("err") <= 50).cast("double")).alias("hit_50"),
            F.avg((F.col("err") <= 250).cast("double")).alias("hit_250"),
            F.expr("percentile_approx(err_prior, 0.5)").alias("prior_median_km"),
            F.avg((F.col("err_prior") <= 50).cast("double")).alias("prior_hit_50")]
    return (df.groupBy(*by).agg(*aggs) if by else df.agg(*aggs)).toPandas()
TRUTH = acc(EST.withColumn("pin", F.when(F.col("pin_representative"), "representative (≤ 25 km)")
                                   .when(F.col("pin_representative").isNotNull(), "not representative").otherwise("no flow centre")), ["pin"])
FOOT = acc(EST.withColumn("footprint", F.when(F.col("state").isin(CORE), "in core states").otherwise("outside core states"))
              .withColumn("pin", F.when(F.col("pin_representative"), "representative").otherwise("all other")), ["footprint", "pin"])
STAT = acc(EST, ["geo_status"])          # a shared building that scores far better than ok would be manufacturing hits
TRUTH.to_csv(f"{OUT_DIR}/truth_sensitivity.csv", index=False)
FOOT.to_csv(f"{OUT_DIR}/footprint_bias.csv", index=False)

# error vs k: fixed cohort (targets with ≥ K_MAX usable neighbours), tightest-first vs random order
coh = SP.groupBy("target").count().where(F.col("count") >= K_MAX).select("target")
SPc = SP.join(coh, "target").withColumn("rnd", F.xxhash64(F.concat(F.col("j"), F.col("target"), F.lit(str(SEED)))))
curves = []
for order_, name_ in (("w", "tightest_first"), ("rnd", "random")):
    r = (SPc.withColumn("_rk", F.row_number().over(Window.partitionBy("target").orderBy(order_, "j"))).where(F.col("_rk") <= K_MAX)
            .withColumn("iv", 1 / F.col("w") ** 2))
    wc = Window.partitionBy("target").orderBy("_rk").rowsBetween(Window.unboundedPreceding, 0)
    r = (r.withColumn("sx", F.sum(F.col("iv") * F.col("cx")).over(wc)).withColumn("sy", F.sum(F.col("iv") * F.col("cy")).over(wc))
          .withColumn("sz", F.sum(F.col("iv") * F.col("cz")).over(wc))
          .withColumn("_n", F.sqrt(F.col("sx") ** 2 + F.col("sy") ** 2 + F.col("sz") ** 2))
          .withColumn("e", chord_km(F.col("sx") / F.col("_n"), F.col("sy") / F.col("_n"), F.col("sz") / F.col("_n"), F.col("tx"), F.col("ty"), F.col("tz"))))
    curves.append(r.groupBy(F.col("_rk").alias("k")).agg(F.expr("percentile_approx(e, 0.5)").alias("median_km"),
                                                          F.avg((F.col("e") <= 50).cast("double")).alias("hit_50"),
                                                          F.count("*").alias("targets"))
                   .withColumn("order", F.lit(name_)).toPandas())
EVK = pd.concat(curves).sort_values(["order", "k"])
EVK.to_csv(f"{OUT_DIR}/error_vs_k.csv", index=False)

# gate: for each predicted-width cut (bin upper edges), the share of targets flagged and hit@50 among them
GATE = (EST.groupBy("wbin").agg(F.count("*").alias("n"), F.sum((F.col("err") <= 50).cast("long")).alias("hits"),
                                F.max("pred_width_km").alias("width_to_km")).orderBy("wbin").toPandas())
GATE["flagged_share"] = GATE["n"].cumsum() / GATE["n"].sum()
GATE["precision_hit50"] = GATE["hits"].cumsum() / GATE["n"].cumsum()
_ok = GATE[GATE.precision_hit50 >= GATE_PRECISION]
GATE_WIDTH = float(_ok.width_to_km.max()) if len(_ok) else None
GATE.to_csv(f"{OUT_DIR}/gate_operating_points.csv", index=False)

# locating class of each neighbour j: median error of its single (left-out) kernel over the targets it scores
LCLS = (SP.groupBy("j").agg(F.count("*").alias("pairs"), F.expr("percentile_approx(err1, 0.5)").alias("median_err1_km"))
          .where(F.col("pairs") >= CLASS_MIN_PAIRS)
          .withColumn("locating_class", F.when(F.col("median_err1_km") <= CLASS_KM["PIN"], "PIN")
                      .when(F.col("median_err1_km") <= CLASS_KM["METRO"], "METRO").when(F.col("median_err1_km") <= CLASS_KM["REGION"], "REGION")
                      .otherwise("NONE")))
LCLS.write.mode("overwrite").parquet(f"{WORK}/lcls"); LCLS = spark.read.parquet(f"{WORK}/lcls")
REG = (LCLS.join(K.where("is_hub").select(F.col("cust").alias("j"), "degree"), "j")
          .join(spark.table(T_CUST).select(F.trim(F.col("mdm_id")).alias("j"), "naics2", "party_type"), "j", "left")
          .orderBy(F.desc("degree")).toPandas())
REG.to_csv(f"{OUT_DIR}/hub_locating_registry.csv", index=False)

S_all = acc(EST)
rep = TRUTH[TRUTH.pin.str.startswith("representative")]
rep_med = float(rep.median_km.iloc[0]) if len(rep) else None
checks = pd.DataFrame([
    {"check": "beats the footprint prior (median km)", "value": f"{S_all.median_km[0]:,.0f} vs {S_all.prior_median_km[0]:,.0f}",
     "ok": bool(S_all.median_km[0] < S_all.prior_median_km[0])},
    {"check": f"leave-one-out (representative median ≥ {LEAK_FLAG_KM} km)", "value": f"{rep_med:,.1f}" if rep_med is not None else "—",
     "ok": rep_med is None or rep_med >= LEAK_FLAG_KM},
    {"check": f"held-out share inside r90 within {R90_COVER}", "value": f"{chk['in_r90']:.1%} (r50 {chk['in_r50']:.1%}, n {chk['n']:,})",
     "ok": R90_COVER[0] <= (chk["in_r90"] or 0) <= R90_COVER[1]}])
out(sec(f"§4 Self-scoring · window {K_MONTHS[0]} → {K_MONTHS[-1]} · hubs: degree ≥ {HUB_CUT:,.0f} · "
        f"{(time.time() - t0) / 60:.1f} min"),
    tbl(checks),
    sec("Accuracy by ground-truth quality (never quote the pooled number alone)"),
    tbl(TRUTH, pct=("hit_25", "hit_50", "hit_250", "prior_hit_50")),
    sec("Accuracy by address status of the target (shared_building should look like ok, not better)"),
    tbl(STAT, pct=("hit_25", "hit_50", "hit_250", "prior_hit_50")),
    sec(f"Footprint bias · core states: {', '.join(CORE)}"),
    tbl(FOOT, pct=("hit_25", "hit_50", "hit_250", "prior_hit_50")),
    sec("Radius calibration (even half) · evidence class by r50"),
    tbl(CAL, pct=("p_within_50km", "p_within_250km")),
    sec(f"Gate: widest predicted width with hit@50 precision ≥ {GATE_PRECISION:.0%}: "
        + (f"{GATE_WIDTH:,.1f} km" if GATE_WIDTH is not None else "none")),
    tbl(GATE[["wbin", "width_to_km", "n", "flagged_share", "precision_hit50"]], pct=("flagged_share", "precision_hit50")),
    sec(f"Error vs k, median km (fixed cohort: {int(EVK.targets.max()) if len(EVK) else 0:,} targets with ≥ {K_MAX} usable neighbours)"),
    tbl(EVK.pivot(index="k", columns="order", values="median_km").reset_index()),
    sec(f"Hub locating registry · {len(REG):,} hubs with ≥ {CLASS_MIN_PAIRS} scored pairs"),
    tbl(REG.groupby("locating_class").agg(hubs=("j", "count"), median_err_km=("median_err1_km", "median")).reset_index()
        if len(REG) else pd.DataFrame()))
_fails = checks[~checks.ok]
assert _fails.empty, "run check failed: " + "; ".join(_fails.check)

## §5 `locate()`: outside counterparties and customers without a usable address; publish

Outside entities: neighbours = the customers that paid them or were paid by them in the window (ids → `ent_l2` of the
entity map). Customers without a usable address: neighbours = the customers they transacted with. Neither target is
inside a neighbour's cloud (the cloud holds located customers only), so no leave-one-out is needed. A target with no
usable neighbour is `NO_EVIDENCE` (kept, never filled). `locate_ids()` and `locate_neighbours()` stay defined for ad
hoc use (an entity id, or a list of PNC customers for a counterparty not in the graph).

**v1.2 — calibration of outside entities.** The customer calibration was ~30 points too confident for outside
entities (validation 2026-11-26: PIN realised ~58% within 50 km, not 90%; the strict leave-one-out showed the customer
calibration itself is sound). Radii, probabilities and evidence class of `pkg_cpty_geo` now come from outside entities
that are the same business as a PNC organisation customer: unique cleaned name on both sides, and either the identical
full name or the customer traded with the entity. Their location is re-estimated with that customer left out (never a
neighbour, removed from every neighbour's cloud) and scored against its address. Bins: number of PNC neighbours
(`OUT_NB_BUCKETS`) × predicted width (`OUT_BINS`, at least `OUT_MIN_PER_BIN` pairs a bin); fitted on one hash half,
checked on the other. `evidence_class_customer_cal` keeps the old class for comparison. `pkg_cust_geo` (`inf_*`) keeps
the customer calibration: its targets are customers.

In [ ]:
# §5  locate() and publish
t0 = time.time()
K = spark.read.parquet(f"{WORK}/k")
KU = usable(K.select(F.col("cust").alias("j"), F.col("k_sx").alias("kx"), F.col("k_sy").alias("ky"), F.col("k_sz").alias("kz"),
                     F.col("k_n_cp").alias("kn"))).select("j", "cx", "cy", "cz", "w")
KU.write.mode("overwrite").parquet(f"{WORK}/ku"); KU = spark.read.parquet(f"{WORK}/ku")

def locate_pairs(P):
    """P: target, j → calibrated estimate per target (est_lat/lon, cell, radius, probabilities, evidence class)."""
    E = combine(P.join(KU, "j"))
    E = calibrate(E)
    E = add_cell(E, "est_lat", "est_lon")
    return E.select("target", "est_lat", "est_lon", "cell_zip3", "cell_state", "pred_width_km", "r50_km", "r90_km",
                    "p_within_50km", "p_within_250km", "k_used", "tightest_bw_km", "evidence_class")

def locate_neighbours(neigh):
    """Ad hoc: {"label": [mdm_id, ...]} → pandas estimates (a counterparty outside the graph)."""
    rows = [(k_, str(j)) for k_, js in neigh.items() for j in js]
    P = spark.createDataFrame(pd.DataFrame(rows, columns=["target", "j"]))
    return locate_pairs(P).toPandas()

# outside entities
ent_map = spark.table(T_MAP).select(F.col("cpty_id").alias("other"), F.col("ent_l2").alias("target"))
PE = (spark.read.parquet(f"{WORK}/wp").where(F.col("okind") == "CPTY").join(ent_map, "other")
          .select("target", F.col("cust").alias("j")).distinct())
PE.write.mode("overwrite").parquet(f"{WORK}/pe"); PE = spark.read.parquet(f"{WORK}/pe")
CE = locate_pairs(PE)
CE.write.mode("overwrite").parquet(f"{WORK}/ce"); CE = spark.read.parquet(f"{WORK}/ce")
ents = PE.groupBy("target").agg(F.countDistinct("j").alias("n_neighbours"))

# ---- v1.2: view-matched calibration of outside entities ---------------------------------------
def nb_bucket(c):
    e = F.lit(str(OUT_NB_BUCKETS[0]))
    for i, lo in enumerate(OUT_NB_BUCKETS):
        hi = OUT_NB_BUCKETS[i + 1] if i + 1 < len(OUT_NB_BUCKETS) else None
        lab = (f"{lo}" if hi == lo + 1 else (f"{lo}-{hi - 1}" if hi else f"{lo}+"))
        e = F.when(c >= lo, F.lit(lab)).otherwise(e)
    return e
_legal = "|".join(sorted((w.replace(" ", r"\s") for w in LEGAL_WORDS), key=len, reverse=True))
def _key(c):
    s_ = F.regexp_replace(F.upper(F.coalesce(c, F.lit(""))), "&", " AND ")
    s_ = F.regexp_replace(s_, r"[^A-Z0-9 ]", " ")
    s_ = F.regexp_replace(s_, rf"(^| )({_legal})(?= |$)", " ")
    return F.trim(F.regexp_replace(s_, r"\s+", " "))
def _plain(c):
    s_ = F.regexp_replace(F.upper(F.coalesce(c, F.lit(""))), "&", " AND ")
    return F.trim(F.regexp_replace(F.regexp_replace(s_, r"[^A-Z0-9 ]", " "), r"\s+", " "))

CAL_OUT_INFO, OCHK, OCAL = None, pd.DataFrame(), pd.DataFrame()
if OUTSIDE_CALIBRATION:
    ccols = dict(spark.table(T_CUST).dtypes)
    e_ = (spark.table(T_ENT).where(F.col("name_class").isin(MATCH_NAME_CLASSES))
            .select(F.col("ent_id").alias("target"), "ent_name").withColumn("key", _key(F.col("ent_name")))
            .where(F.length(F.regexp_replace("key", " ", "")) >= KEY_MIN_CHARS))
    e_ = e_.join(e_.groupBy("key").count().where("count = 1").select("key"), "key", "left_semi")
    cf = F.col("party_type") == "O"
    if "is_pnc_own" in ccols:
        cf = cf & ~F.coalesce(F.col("is_pnc_own").cast("boolean"), F.lit(False))
    if "name_group" in ccols:
        cf = cf & F.col("name_group").isNull()
    c_ = (spark.table(T_CUST).where(cf).select(F.trim(F.col("mdm_id")).alias("c"), "customer_name")
            .withColumn("key", _key(F.col("customer_name"))).where(F.length(F.regexp_replace("key", " ", "")) >= KEY_MIN_CHARS))
    c_ = c_.join(c_.groupBy("key").count().where("count = 1").select("key"), "key", "left_semi")
    truth = (spark.read.parquet(f"{WORK}/g0").where(F.col("geo_status").isin(TRUTH_STATUSES))
               .select(F.col("mdm_id").alias("c"), F.col("ux").alias("tx"), F.col("uy").alias("ty"), F.col("uz").alias("tz")))
    direct = PE.select("target", F.col("j").alias("c")).distinct().withColumn("direct", F.lit(True))
    MT = (e_.join(c_, "key").join(truth, "c").join(direct, ["target", "c"], "left")
            .withColumn("direct", F.coalesce("direct", F.lit(False)))
            .withColumn("exact", _plain(F.col("ent_name")) == _plain(F.col("customer_name")))
            .where(F.col("direct") | F.col("exact"))
            .select("target", "c", "tx", "ty", "tz", "direct", "exact"))
    MT.write.mode("overwrite").parquet(f"{WORK}/mt"); MT = spark.read.parquet(f"{WORK}/mt")
    # leave-one-out estimates: matched customer never a neighbour, and taken out of every neighbour's cloud
    NBm = PE.join(MT.select("target", "c", "tx", "ty", "tz"), "target").where(F.col("j") != F.col("c"))
    jc = (spark.read.parquet(f"{WORK}/wp").where(F.col("okind") == "CUST")
            .select(F.col("cust").alias("j"), F.col("other").alias("c")).distinct().withColumn("has_c", F.lit(True)))
    Kr = K.select(F.col("cust").alias("j"), "k_sx", "k_sy", "k_sz", "k_n_cp")
    Pm = (NBm.join(Kr, "j").join(jc, ["j", "c"], "left").withColumn("has_c", F.coalesce("has_c", F.lit(False)))
            .select("target", "j",
                    F.when(F.col("has_c"), F.col("k_sx") - F.col("tx")).otherwise(F.col("k_sx")).alias("kx"),
                    F.when(F.col("has_c"), F.col("k_sy") - F.col("ty")).otherwise(F.col("k_sy")).alias("ky"),
                    F.when(F.col("has_c"), F.col("k_sz") - F.col("tz")).otherwise(F.col("k_sz")).alias("kz"),
                    F.when(F.col("has_c"), F.col("k_n_cp") - 1).otherwise(F.col("k_n_cp")).alias("kn")))
    Em = combine(usable(Pm).select("target", "j", "cx", "cy", "cz", "w"))
    nbm = NBm.groupBy("target").agg(F.countDistinct("j").alias("nb"))
    CAL_SET = (Em.join(nbm, "target").join(MT.select("target", "tx", "ty", "tz", "direct"), "target")
                 .withColumn("err", chord_km(F.col("ex"), F.col("ey"), F.col("ez"), F.col("tx"), F.col("ty"), F.col("tz")))
                 .withColumn("bucket", nb_bucket(F.col("nb")))
                 .withColumn("half", F.abs(F.xxhash64(F.concat(F.col("target"), F.lit(str(SEED))))) % 2)
                 .select("target", "pred_width_km", "err", "bucket", "half", "direct").toPandas())
    CAL_SET.to_csv(f"{OUT_DIR}/outside_calibration_pairs.csv", index=False)
    n_fit = int((CAL_SET.half == 0).sum())
    if n_fit < OUT_MIN_TOTAL:
        CAL_OUT_INFO = f"only {n_fit:,} calibration pairs (< {OUT_MIN_TOTAL:,}): customer calibration kept for pkg_cpty_geo"
    else:
        def fit(d):
            nb_ = max(1, min(OUT_BINS, len(d) // OUT_MIN_PER_BIN))
            cuts = sorted(set(np.quantile(d.pred_width_km, [i / nb_ for i in range(1, nb_)]))) if nb_ > 1 else []
            b_ = np.searchsorted(np.asarray(cuts), d.pred_width_km.to_numpy(), side="left")
            rows_ = []
            for i in range(len(cuts) + 1):
                x_ = d.err.to_numpy()[b_ == i]
                if len(x_) == 0:
                    continue
                rows_.append({"bin": i, "width_to_km": cuts[i] if i < len(cuts) else np.inf, "n": len(x_),
                              "r50_km": float(np.quantile(x_, 0.5)), "r90_km": float(np.quantile(x_, 0.9)),
                              "p_within_50km": float((x_ <= 50).mean()), "p_within_250km": float((x_ <= 250).mean())})
            t_ = pd.DataFrame(rows_)
            t_["r50_km"] = t_.r50_km.cummax(); t_["r90_km"] = t_.r90_km.cummax()
            t_["p_within_50km"] = t_.p_within_50km[::-1].cummax()[::-1]; t_["p_within_250km"] = t_.p_within_250km[::-1].cummax()[::-1]
            return t_
        fit0 = CAL_SET[CAL_SET.half == 0]
        parts_ = []
        for bk, d in fit0.groupby("bucket"):
            if len(d) >= OUT_MIN_PER_BIN:
                parts_.append(fit(d).assign(bucket=bk))
        pooled = fit(fit0).assign(bucket="pooled")
        OCAL = pd.concat(parts_ + [pooled], ignore_index=True)
        OCAL["evidence_class"] = np.select([OCAL.r50_km <= CLASS_KM["PIN"], OCAL.r50_km <= CLASS_KM["METRO"],
                                            OCAL.r50_km <= CLASS_KM["REGION"]], ["PIN", "METRO", "REGION"], "NONE")
        OCAL.to_csv(f"{OUT_DIR}/radius_calibration_outside.csv", index=False)
        have = set(OCAL.bucket) - {"pooled"}
        def assign(d):
            out_ = []
            for bk, g in d.groupby("bucket"):
                t_ = OCAL[OCAL.bucket == (bk if bk in have else "pooled")].sort_values("bin")
                idx = np.searchsorted(t_.width_to_km.to_numpy(), g.pred_width_km.to_numpy(), side="left")
                idx = np.minimum(idx, len(t_) - 1)
                out_.append(g.assign(**{c_: t_[c_].to_numpy()[idx] for c_ in ("r50_km", "r90_km", "p_within_50km", "evidence_class")}))
            return pd.concat(out_)
        chk = assign(CAL_SET[CAL_SET.half == 1])
        OCHK = (chk.assign(in50=chk.err <= chk.r50_km, in90=chk.err <= chk.r90_km, hit50=chk.err <= 50)
                   .groupby(["evidence_class"]).agg(entities=("target", "count"), inside_r50=("in50", "mean"),
                                                    inside_r90=("in90", "mean"), hit_50=("hit50", "mean"),
                                                    predicted_p50=("p_within_50km", "mean"), median_km=("err", "median"))
                   .reset_index())
        OCHK["_o"] = OCHK.evidence_class.map({"PIN": 0, "METRO": 1, "REGION": 2, "NONE": 3}); OCHK = OCHK.sort_values("_o").drop(columns="_o")
        OCHK.to_csv(f"{OUT_DIR}/outside_calibration_check.csv", index=False)
        CAL_OUT_INFO = (f"{len(CAL_SET):,} calibration entities ({int(CAL_SET.direct.sum()):,} traded with their matched customer), "
                        f"fit on {n_fit:,}, checked on {len(CAL_SET) - n_fit:,}")
        # apply to every outside entity with an estimate
        CEo = CE.join(ents, "target").withColumn("bucket", nb_bucket(F.col("n_neighbours")))
        rows_ = []
        for bk in sorted(set(OCAL.bucket)):
            t_ = OCAL[OCAL.bucket == bk].sort_values("bin")
            for _, r_ in t_.iterrows():
                rows_.append((bk, int(r_.bin), float(r_.width_to_km) if np.isfinite(r_.width_to_km) else 1e18, float(r_.r50_km),
                              float(r_.r90_km), float(r_.p_within_50km), float(r_.p_within_250km), r_.evidence_class))
        OT = pd.DataFrame(rows_, columns=["cbucket", "obin", "width_to_km", "o_r50", "o_r90", "o_p50", "o_p250", "o_class"])
        bexpr = F.when(F.col("bucket").isin(list(have)), F.col("bucket")).otherwise(F.lit("pooled"))
        CEo = CEo.withColumn("cbucket", bexpr)
        binexpr = None
        for bk in sorted(set(OT.cbucket)):
            t_ = OT[OT.cbucket == bk].sort_values("obin")
            e_b = F.lit(int(t_.obin.iloc[-1]))
            for _, r_ in t_.iloc[::-1].iterrows():
                e_b = F.when(F.col("pred_width_km") <= float(r_.width_to_km), F.lit(int(r_.obin))).otherwise(e_b)
            binexpr = F.when(F.col("cbucket") == bk, e_b) if binexpr is None else binexpr.when(F.col("cbucket") == bk, e_b)
        CEo = CEo.withColumn("obin", binexpr).join(F.broadcast(spark.createDataFrame(OT.drop(columns="width_to_km"))), ["cbucket", "obin"], "left")
        CE = (CEo.withColumn("evidence_class_customer_cal", F.col("evidence_class"))
                 .withColumn("r50_km", F.col("o_r50")).withColumn("r90_km", F.col("o_r90"))
                 .withColumn("p_within_50km", F.col("o_p50")).withColumn("p_within_250km", F.col("o_p250"))
                 .withColumn("evidence_class", F.col("o_class"))
                 .drop("n_neighbours", "bucket", "cbucket", "obin", "o_r50", "o_r90", "o_p50", "o_p250", "o_class"))
if "evidence_class_customer_cal" not in CE.columns:
    CE = CE.withColumn("evidence_class_customer_cal", F.col("evidence_class"))
CE = CE.withColumn("calibration", F.lit("outside entities" if (OUTSIDE_CALIBRATION and len(OCAL)) else "customers"))
hub_t = (spark.table(T_CPM).where(F.col("month") == HUB_MONTH).select(F.col("ent_id").alias("target"), "hub_tier")
         if HUB_MONTH else ents.select("target", F.lit(None).cast("int").alias("hub_tier")).limit(0))
CP = (ents.join(CE, "target", "left")
          .join(spark.table(T_ENT).select(F.col("ent_id").alias("target"), "name_class"), "target", "left")
          .join(hub_t, "target", "left")
          .withColumn("evidence_class", F.coalesce(F.col("evidence_class"), F.lit("NO_EVIDENCE")))
          .withColumn("evidence_class_customer_cal", F.coalesce(F.col("evidence_class_customer_cal"), F.lit("NO_EVIDENCE")))
          .withColumn("window_end_month", F.lit(LAST))
          .withColumnRenamed("target", "ent_id"))

# customers without a usable address (inferred, separate columns)
G0 = spark.read.parquet(f"{WORK}/g0")
PC = (spark.read.parquet(f"{WORK}/wp").where(F.col("okind") == "CUST")
          .join(G0.where(~F.col("geo_ok")).select(F.col("mdm_id").alias("cust")), "cust", "left_semi")
          .select(F.col("cust").alias("target"), F.col("other").alias("j")).distinct())
CI = locate_pairs(PC).select(F.col("target").alias("mdm_id"), *[F.col(a).alias(b) for a, b in (
    ("est_lat", "inf_lat"), ("est_lon", "inf_lon"), ("cell_zip3", "inf_zip3"), ("cell_state", "inf_state"),
    ("r50_km", "inf_r50_km"), ("r90_km", "inf_r90_km"), ("p_within_50km", "inf_p50km"), ("evidence_class", "inf_class"))])

K_ = spark.read.parquet(f"{WORK}/k").select(F.col("cust").alias("mdm_id"), "k_sx", "k_sy", "k_sz", "k_sw", "k_n_cp", "k_width_km", "is_hub", "degree")
CG = (G0.drop("ux", "uy", "uz")
        .join(spark.read.parquet(f"{WORK}/rvf").withColumnRenamed("cust", "mdm_id"), "mdm_id", "left")
        .withColumn("pin_representative", F.col("geo_registered_vs_flow_km") <= REP_GAP_KM)
        .join(K_, "mdm_id", "left")
        .join(spark.read.parquet(f"{WORK}/lcls").select(F.col("j").alias("mdm_id"), "locating_class"), "mdm_id", "left")
        .join(CI, "mdm_id", "left")
        .withColumn("is_hub", F.coalesce(F.col("is_hub"), F.lit(False)))
        .withColumn("cell_zip3", F.when(F.col("geo_ok"), F.col("zip3")).otherwise(F.col("inf_zip3")))
        .withColumn("cell_state", F.when(F.col("geo_ok"), F.col("state")).otherwise(F.col("inf_state"))))
CG_COLS = ["mdm_id", "party_type", "is_tm", "naics2", "lat", "lon", "zip5", "zip3", "state", "country", "city",
           "addr_loc_rec_type", "addr_cleansed_date", "snapshot_dt", "geo_status", "geo_ok", "coloc_cluster_size",
           "coloc_n_zip5", "coord_decimals", "flag_po_box", "flag_reg_agent", "flag_care_of", "flag_pnc", "moved_12m", "prev_zip3",
           "move_km", "geo_registered_vs_flow_km", "pin_representative", "k_sx", "k_sy", "k_sz", "k_sw", "k_n_cp",
           "k_width_km", "locating_class", "is_hub", "degree", "inf_lat", "inf_lon", "inf_zip3", "inf_state",
           "inf_r50_km", "inf_r90_km", "inf_p50km", "inf_class", "cell_zip3", "cell_state"]
CP_COLS = ["ent_id", "est_lat", "est_lon", "cell_zip3", "cell_state", "pred_width_km", "r50_km", "r90_km", "p_within_50km",
           "p_within_250km", "k_used", "n_neighbours", "tightest_bw_km", "evidence_class", "evidence_class_customer_cal",
           "calibration", "window_end_month", "name_class", "hub_tier"]

def publish(df, k):
    """Write to a staging folder, swap it in, (re)create the external table if the columns changed."""
    t, loc = OUT_TABLES[k], LOC[k]
    stg = f"{WORK}/stage_{k}"
    df.write.mode("overwrite").parquet(stg)
    delete(loc)
    assert rename(stg, loc), f"rename {stg} → {loc} failed"
    sch = spark.read.parquet(loc).schema
    if table_exists(t) and spark.table(t).columns != [f.name for f in sch.fields]:
        spark.sql(f"DROP TABLE {t}")
    cols = ", ".join(f"`{f.name}` {f.dataType.simpleString()}" for f in sch.fields)
    spark.sql(f"CREATE EXTERNAL TABLE IF NOT EXISTS {t} ({cols}) STORED AS PARQUET LOCATION '{loc}'")
    spark.sql(f"REFRESH TABLE {t}")
    return spark.table(t)

cg = publish(CG.select(*CG_COLS), "cust")
cp = publish(CP.select(*CP_COLS), "cpty")
EC = (cp.groupBy("evidence_class").agg(F.count("*").alias("entities")).toPandas()
        .merge(cp.groupBy(F.col("evidence_class_customer_cal").alias("evidence_class")).agg(F.count("*").alias("with_customer_calibration")).toPandas(),
               on="evidence_class", how="outer").fillna(0))
EC["share"] = EC.entities / EC.entities.sum()
EC = EC[["evidence_class", "entities", "share", "with_customer_calibration"]]
EC["entities"] = EC.entities.astype(int); EC["with_customer_calibration"] = EC.with_customer_calibration.astype(int)
IC = (cg.where(~F.col("geo_ok")).groupBy(F.coalesce("inf_class", F.lit("NO_EVIDENCE")).alias("inf_class"))
        .agg(F.count("*").alias("customers")).toPandas())
LC = cg.groupBy("locating_class").agg(F.count("*").alias("customers"), F.sum(F.col("is_hub").cast("int")).alias("hubs")).toPandas()
out(sec(f"§5 Published · {(time.time() - t0) / 60:.1f} min"),
    sec("Calibration of outside entities (v1.2)"),
    para(CAL_OUT_INFO or "OUTSIDE_CALIBRATION = False: customer calibration used",
         "#a00" if (CAL_OUT_INFO or "").startswith("only") else None),
    sec("Held-out check (the half not used to fit): inside_r50 near 50%, inside_r90 near 90%, hit_50 near predicted_p50"),
    tbl(OCHK, pct=("inside_r50", "inside_r90", "hit_50", "predicted_p50")) if len(OCHK) else "",
    sec(f"{OUT_TABLES['cpty']}: outside entities by evidence class (outside calibration; customer calibration for comparison)"),
    tbl(EC.sort_values("entities", ascending=False), pct=("share",)),
    sec(f"{OUT_TABLES['cust']}: customers without a usable address, by inferred class"),
    tbl(IC.sort_values("customers", ascending=False)),
    sec("Customers by locating class (as a neighbour)"),
    tbl(LC.sort_values("customers", ascending=False)),
    para("Impala: INVALIDATE METADATA " + "; INVALIDATE METADATA ".join(OUT_TABLES.values()) + ";"))

## §6 Housekeeping

Work files of this run are deleted; the run log stays in `OUT_DIR`.

In [ ]:
# §6  Housekeeping
_sz = size_gb(WORK)
for _x in ("g0", "geo", "wp", "k", "sp", "rvf", "est", "lcls", "ku", "pe"):
    delete(f"{WORK}/{_x}")
out(sec(f"§6 Work files deleted ({_sz:,.1f} GB) · output: "
        + ", ".join(f"{k} {size_gb(LOC[k]):,.1f} GB" for k in LOC)))